## Task 1: Problem Identification

### Problem Type: Image Classification

This is an **image classification** problem. Each image shows a product surface and we need to assign it one label from 4 possible classes: normal, scratch, dent, or stain.

**Why not the other options?**
- **Object detection** — not needed here. We don't have to find *where* the defect is, just *what* kind it is.
- **Semantic segmentation** — that would label every single pixel. Way overkill for just classifying the whole image.
- **Instance segmentation** — even more complex. Not required at all for this task.

Image classification is the right fit because we just need one label per image. CNNs are really good at picking up visual patterns like scratches, dents, and stains, which makes them a great choice here.

## Task 2: Dataset Exploration

In [ ]:
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

# create output folders
os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('Setup done!')

In [ ]:
data_dir = 'part_2_cnn_computer_vision'
df = pd.read_csv(os.path.join(data_dir, 'labels.csv'))
classes = ['normal', 'scratch', 'dent', 'stain']
img_size = (96, 96)

print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)
print(f'Total images      : {len(df)}')
print(f'Number of classes : {len(classes)}')
print(f'Classes           : {classes}')
print(f'Image dimensions  : 96 x 96 x 3 (RGB)')
print()
print('Images per class:')
for cls in classes:
    count = len(df[df['class'] == cls])
    print(f'  {cls:<10}: {count} images')
print()
print('Class imbalance: None — perfectly balanced dataset')

In [ ]:
# show 5 sample images from each class
# this is just for exploration — not saving this plot
colors = {'normal': '#4CAF50', 'scratch': '#F44336', 'dent': '#FF9800', 'stain': '#9C27B0'}

fig, axes = plt.subplots(4, 5, figsize=(14, 12))

for row, cls in enumerate(classes):
    samples = df[df['class'] == cls]['filename'].tolist()[:5]
    for col, fname in enumerate(samples):
        img = Image.open(os.path.join(data_dir, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.upper(), fontsize=11, fontweight='bold', color=colors[cls])
        else:
            axes[row, col].set_title(f'Sample {col + 1}', fontsize=9, color='gray')

plt.suptitle('Sample Images — All 4 Classes (5 samples each)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# class distribution bar chart
counts = df['class'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(classes, [counts[c] for c in classes], color=[colors[c] for c in classes])
ax.set_title('Number of images per class')
ax.set_ylabel('Count')
ax.set_xlabel('Class')
for i, cls in enumerate(classes):
    ax.text(i, counts[cls] + 1, str(counts[cls]), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Task 3: Image Preprocessing

In [ ]:
# load all images into memory and convert to numpy arrays
def load_imgs(df, base_dir, size=(96, 96)):
    cls_map = {c: i for i, c in enumerate(classes)}
    imgs, labels = [], []

    for _, row in df.iterrows():
        path = os.path.join(base_dir, row['filename'])
        img = Image.open(path).convert('RGB').resize(size)
        imgs.append(np.array(img))
        labels.append(cls_map[row['class']])

    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int32)

print('Loading images...')
X, y = load_imgs(df, data_dir)
print(f'X shape : {X.shape}  (images, height, width, channels)')
print(f'y shape : {y.shape}')
print(f'Pixel range before normalisation: [{X.min():.0f}, {X.max():.0f}]')

# normalise pixel values from 0-255 to 0-1
X = X / 255.0
print(f'Pixel range after normalisation : [{X.min():.2f}, {X.max():.2f}]')

In [ ]:
# split into training (80%) and testing (20%)
# stratify=y keeps the same class balance in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set  : {X_train.shape[0]} images')
print(f'Testing set   : {X_test.shape[0]} images')
print()
print(f'Train class counts: {np.bincount(y_train)}')
print(f'Test class counts : {np.bincount(y_test)}')

In [ ]:
# data augmentation — applies random transformations during training
# helps the model see more variation and avoid overfitting
aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomFlip('vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
], name='augmentation')

# show what augmentation does to one sample image
sample = X_train[0:1]
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(sample[0])
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')
for i in range(1, 6):
    out = aug(sample, training=True)[0].numpy()
    axes[i].imshow(out)
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Augmentation examples (same image, 5 random variations)', fontsize=10)
plt.tight_layout()
plt.show()

## Task 4: CNN Model Creation

**Architecture overview:**

- **Augmentation layer** — random flips, rotations, zoom (only active during training)
- **Block 1** — Conv2D(32) + ReLU → MaxPooling: learns basic edges and textures
- **Block 2** — Conv2D(64) + ReLU → MaxPooling: learns more complex shapes
- **Block 3** — Conv2D(128) + ReLU → MaxPooling: learns high-level features
- **Flatten** — converts 3D feature maps into a 1D vector
- **Dense(256) + Dropout(0.4)** — fully connected layer with regularisation to reduce overfitting
- **Dense(4, Softmax)** — output layer giving probability for each of the 4 classes

In [ ]:
def make_cnn(input_shape=(96, 96, 3), n_classes=4):
    model = keras.Sequential(name='defect_classifier')

    # augmentation (only applied during training, not during evaluation)
    model.add(aug)

    # block 1 — 32 filters, learns basic patterns
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))

    # block 2 — 64 filters, learns more detail
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # block 3 — 128 filters, learns complex features
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # classification head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.4))  # dropout prevents overfitting
    model.add(layers.Dense(n_classes, activation='softmax'))  # 4 class probabilities

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn = make_cnn()
cnn.summary()

## Task 5: Model Training and Evaluation

In [ ]:
# callbacks to make training smarter
cb = [
    # stop early if val_loss doesn't improve for 10 epochs
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # reduce learning rate if val_loss gets stuck
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

hist = cnn.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=cb,
    verbose=1
)

In [ ]:
# plot training vs validation loss and accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist.history['loss'], label='Train loss')
axes[0].plot(hist.history['val_loss'], label='Val loss', linestyle='--')
axes[0].set_title('Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(hist.history['accuracy'], label='Train accuracy')
axes[1].plot(hist.history['val_accuracy'], label='Val accuracy', linestyle='--')
axes[1].set_title('Accuracy over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved -> results/accuracy_loss_curves.png')

In [ ]:
# evaluate model on train and test sets
tr_loss, tr_acc = cnn.evaluate(X_train, y_train, verbose=0)
te_loss, te_acc = cnn.evaluate(X_test,  y_test,  verbose=0)

probs = cnn.predict(X_test, verbose=0)
preds = np.argmax(probs, axis=1)

print(f'Training accuracy : {tr_acc*100:.2f}%  |  Loss: {tr_loss:.4f}')
print(f'Testing  accuracy : {te_acc*100:.2f}%  |  Loss: {te_loss:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, preds, target_names=classes))

In [ ]:
# confusion matrix — shows where the model gets confused
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved -> results/confusion_matrix.png')

In [ ]:
# show 12 sample predictions from the test set
idx_map = {i: c for i, c in enumerate(classes)}
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
axes = axes.flatten()

for i, idx in enumerate(sample_idx):
    true_cls = idx_map[y_test[idx]]
    pred_cls = idx_map[preds[idx]]
    conf     = probs[idx][preds[idx]] * 100
    correct  = true_cls == pred_cls

    axes[i].imshow(X_test[idx])
    axes[i].axis('off')
    axes[i].set_title(
        f'True: {true_cls}\nPred: {pred_cls} ({conf:.1f}%)\n{"correct" if correct else "wrong"}',
        fontsize=9,
        color='green' if correct else 'red'
    )

plt.suptitle('Sample Predictions on Test Set', fontsize=12)
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved -> sample_predictions/prediction_outputs.png')

## Task 6: CNN Concept Explanation

### What is Convolution?

A convolution is basically a small filter (like 3x3 pixels) that slides across the image step by step. At each position, it multiplies the filter values with the image pixels underneath and adds them up. This produces a feature map that shows where certain patterns appear in the image. Early layers tend to detect simple things like edges and corners, while deeper layers pick up more complex patterns like shapes and textures.

---

### Why is Pooling used?

MaxPooling takes a small region (like 2x2 pixels) and keeps only the maximum value. This shrinks the feature maps — for example, a 96x96 image becomes 48x48 after one pooling layer. It does two useful things: it reduces the number of computations going forward, and it makes the model less sensitive to small shifts in position (so if a scratch moves slightly, the model still recognises it).

---

### Why ReLU?

ReLU stands for Rectified Linear Unit. It just does `max(0, x)` — anything negative becomes 0, positive values stay as they are. We need non-linear activation functions so the network can learn complex patterns, not just straight lines. ReLU is fast and simple, and it avoids the vanishing gradient problem that sigmoid or tanh can cause in deeper networks.

---

### Why CNNs and not regular neural networks for images?

A regular Dense (fully connected) network flattens the image into a 1D array first, which loses all the spatial information — it no longer knows which pixels are next to each other. On top of that, a 96x96 image has 27,648 values. Just connecting that to 256 neurons would be about 7 million parameters in the very first layer alone.

CNNs solve both problems. They keep the 2D structure of the image intact, and they use shared filters — the same small filter slides over the whole image — so the number of parameters stays very manageable. They're also designed to learn spatial features like edges, shapes, and textures, which is exactly what image classification needs.

## Task 7: Business Use Case Mapping

### Manufacturing Quality Inspection

The most direct real-world application of this model is automated quality control on a factory production line. A camera takes pictures of each product as it moves along the conveyor belt, and the CNN classifies whether it looks normal or has a defect (scratch, dent, or stain). Defective items can be flagged and removed without any human needing to check each one manually.

**How it would work in practice:**
1. Camera takes an image of each product on the line
2. CNN classifies it as normal, scratch, dent, or stain
3. If defective, a signal triggers automatic rejection
4. Data gets logged for quality reports and trend analysis

**Other domains where this type of CNN can be applied:**

- **Healthcare** — identifying tumors or abnormalities in X-rays and MRI scans
- **Agriculture** — spotting diseased or damaged crops from drone or satellite images
- **Retail** — checking if store shelves are properly stocked using CCTV footage
- **Security** — detecting suspicious objects or behaviour in surveillance systems
- **Autonomous vehicles** — recognising road signs, pedestrians, and obstacles in real time

In [ ]:
print('=' * 55)
print('FINAL SUMMARY')
print('=' * 55)
print(f'Problem Type      : Multi-class Image Classification')
print(f'Dataset           : 480 images, 4 classes, 96x96 RGB')
print(f'Model             : 3-block CNN + Dense head')
print(f'Augmentation      : RandomFlip, RandomRotation, RandomZoom')
print(f'Training Accuracy : {tr_acc*100:.2f}%')
print(f'Testing  Accuracy : {te_acc*100:.2f}%')
print()
print('Output files saved:')
print('  results/accuracy_loss_curves.png')
print('  results/confusion_matrix.png')
print('  sample_predictions/prediction_outputs.png')